# 8.3 期货期限结构套利策略

## 学习目标
- 深入理解期限结构在期货交易中的信号意义
- 实现 Calendar Spread（日历价差）策略
- 了解 VIX 期货 Roll Yield 量化策略


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
np.random.seed(42)
print('Libraries loaded')


## 1. 期限结构形状与交易信号

**期限结构**（Term Structure）是不同到期月份期货价格的分布图。

形状告诉我们市场的预期：
- **Contango（上斜）**：市场预期未来价格更高（或储存成本高）
- **Backwardation（下斜）**：市场预期未来价格更低（或需求紧张）
- **套利逻辑**：把握两者之间的转换时机，或利用斜率绝对值进行 Carry 策略


In [ ]:
# 模拟原油期货 12 个月的期限结构（两个场景）
spot = 80  # 现货价格

months = np.arange(1, 13)
# Contango 场景
F_contango = spot * (1 + 0.003)**months + np.random.normal(0, 0.2, 12)
# Backwardation 场景
F_backwardation = spot * (1 - 0.002)**months + np.random.normal(0, 0.2, 12)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(months, F_contango, 'ro-', ms=7, lw=2, label='期货价格')
ax1.axhline(spot, color='gray', linestyle='--', label=f'现货 {spot}')
ax1.set_title('Contango：期货溢价（远月贵）')
ax1.set_xlabel('到期月份'); ax1.set_ylabel('价格')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(months, F_backwardation, 'bo-', ms=7, lw=2)
ax2.axhline(spot, color='gray', linestyle='--', label=f'现货 {spot}')
ax2.set_title('Backwardation：期货贴水（远月便宜）')
ax2.set_xlabel('到期月份')
ax2.legend(); ax2.grid(alpha=0.3)
plt.suptitle('原油期货期限结构对比', fontsize=13)
plt.tight_layout(); plt.show()


## 2. Calendar Spread 策略：看期限结构斜率回归


In [ ]:
# 模拟 Calendar Spread：做空近月 + 做多远月（在 Contango 中）
# 策略收益来自期限结构斜率压缩
np.random.seed(42)
n_months = 60
spread_series = []
for _ in range(n_months):
    # 随机生成当月近/远期价差（均值回归）
    spread = np.random.normal(3, 1.5)  # 正值 = Contango
    spread_series.append(spread)
spread_s = pd.Series(spread_series)

# 策略：当价差过大时做空近月（期待压缩）
signal = pd.Series(0.0, index=spread_s.index)
thresh = spread_s.expanding().mean() + 1.5 * spread_s.expanding().std()
signal[spread_s > thresh] = -1  # 价差过大 → 做空近月（等待收窄）

# 收益：下一期价差变化
spread_change = spread_s.diff().shift(-1)
strat_ret = signal * (-spread_change / 80)  # 归一化为收益率

plt.figure(figsize=(12, 5))
plt.subplot(2,1,1)
spread_s.plot(label='近远月价差')
thresh.plot(label='上轨（做空信号）', linestyle='--', color='red')
plt.title('Calendar Spread 策略信号')
plt.legend(); plt.grid(alpha=0.3)
plt.subplot(2,1,2)
strat_ret.fillna(0).cumsum().plot(label='Calendar Spread 策略累积PNL', color='green')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 🎯 练习

1. 研究 WTI 和 Brent 原油期货间的价差历史变化，设计一个跨品种套利策略。
2. 对 VIX 期货（$VIX3M - VIX）计算展期成本，设计一个做空波动率的 Roll Yield 收益策略。
3. Calendar Spread 策略在自然灾害/地缘冲突导致供应冲击时会如何？请设计风险管理规则。

---
**下一节** → `../09_hft/01_order_book_basics.ipynb`